In [ ]:
!git --version
!git config --global user.name VitorKF
!git config --global user.email vkopacekfernandes@gmail.com
!git config --global init.defaultBranch main
!git config --global --list

git version 2.43.0
user.name=VitorKF
user.email=vkopacekfernandes@gmail.com
init.defaultbranch=main


In [44]:
# Uso da biblioteca pandas
import pandas as pd
import numpy as np

# Nomes dos campos
nomes = ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides',
         'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']

# Carregando o dataset
df = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv',
    skiprows = 1,
    sep =";",
    names=nomes
)

pd.set_option('display.width', 166)

print(df)

      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  alcohol  quality
0               7.4             0.700         0.00             1.9      0.076                 11.0                  34.0  0.99780  3.51       0.56      9.4        5
1               7.8             0.880         0.00             2.6      0.098                 25.0                  67.0  0.99680  3.20       0.68      9.8        5
2               7.8             0.760         0.04             2.3      0.092                 15.0                  54.0  0.99700  3.26       0.65      9.8        5
3              11.2             0.280         0.56             1.9      0.075                 17.0                  60.0  0.99800  3.16       0.58      9.8        6
4               7.4             0.700         0.00             1.9      0.076                 11.0                  34.0  0.99780  3.51       0.56      9.4        5
...       

In [48]:
# Criando a coluna de classificação
def classificar_qualidade(nota):
    if nota <= 4:
        return "Baixa"
    elif nota <= 6:
        return "Média"
    else:
        return "Alta"

df["Class"] = df["quality"].apply(classificar_qualidade)

In [49]:
# Separando o dataset
# X será a matriz de características, sem quality e Class
X = df.drop(columns=['quality', 'Class']).values
print(f'X: {X}')

# y será o vetor com a classificação (target)
y = df['Class'].values
print(f'y: {y}')

X: [[ 7.4    0.7    0.    ...  3.51   0.56   9.4  ]
 [ 7.8    0.88   0.    ...  3.2    0.68   9.8  ]
 [ 7.8    0.76   0.04  ...  3.26   0.65   9.8  ]
 ...
 [ 6.3    0.51   0.13  ...  3.42   0.75  11.   ]
 [ 5.9    0.645  0.12  ...  3.57   0.71  10.2  ]
 [ 6.     0.31   0.47  ...  3.39   0.66  11.   ]]
y: ['Média' 'Média' 'Média' ... 'Média' 'Média' 'Média']


In [20]:
# Importando o classificador KNN
from sklearn.neighbors import KNeighborsClassifier

# Definindo o número de vizinhos como 3
classificador_vinhos = KNeighborsClassifier(n_neighbors=3)

# Treinando o modelo com os dados
classificador_vinhos.fit(X, y)

KNeighborsClassifier(n_neighbors=3)

In [52]:
# Usando validação cruzada para avaliar o modelo
from sklearn.model_selection import cross_val_score

# Dividindo os dados em 5 partes para validação
scores_dt = cross_val_score(
    classificador_vinhos,
    X,
    y,
    scoring='accuracy',
    cv=5
)

# Exibindo a acurácia média
print(scores_dt.mean())

0.8217652821316616


In [53]:
# Treinando e avaliando diferentes números de vizinhos
for i in range(1, 15):
    classificador_vinhos = KNeighborsClassifier(n_neighbors=i)

    classificador_vinhos.fit(X, y)

    scores_dt = cross_val_score(
        classificador_vinhos,
        X,
        y,
        scoring='accuracy',
        cv=5
    )

    print(f'Vizinhos: {i}  Acurácia: {scores_dt.mean()}')

Vizinhos: 1  Acurácia: 0.7461010971786834
Vizinhos: 2  Acurácia: 0.7060697492163011
Vizinhos: 3  Acurácia: 0.7848863636363637
Vizinhos: 4  Acurácia: 0.7773824451410658
Vizinhos: 5  Acurácia: 0.7955211598746081
Vizinhos: 6  Acurácia: 0.7955172413793103
Vizinhos: 7  Acurácia: 0.806773119122257
Vizinhos: 8  Acurácia: 0.810521159874608
Vizinhos: 9  Acurácia: 0.8123922413793105
Vizinhos: 10  Acurácia: 0.8155172413793104
Vizinhos: 11  Acurácia: 0.8167652821316616
Vizinhos: 12  Acurácia: 0.8161402821316613
Vizinhos: 13  Acurácia: 0.8192652821316614
Vizinhos: 14  Acurácia: 0.8123883228840125


In [54]:
# Treinando o modelo KNN
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Escolha inicial do número de vizinhos
knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

# Validação cruzada estratificada com 5 divisões
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Avaliando a acurácia
scores = cross_val_score(
    knn, X, y,
    cv=cv,
    scoring="accuracy"
)

print("Acurácias:", np.round(scores, 4))
print(f"Acurácia média: {scores.mean():.4f}")
print(f"Acurácia média em porcentagem: {scores.mean() * 100:.2f}%")

Acurácias: [0.8125 0.8406 0.8344 0.8156 0.8433]
Acurácia média: 0.8293
Acurácia média em porcentagem: 82.93%


In [55]:
# Testando diferentes números de vizinhos
resultados = []

for k in range(1, 21):
    modelo = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k)
    )

    scores = cross_val_score(
        modelo, X, y,
        cv=cv,
        scoring="accuracy"
    )

    resultados.append({
        "Vizinhos (k)": k,
        "Acurácia média": scores.mean(),
        "Acurácia (%)": scores.mean() * 100
    })

resultados_df = pd.DataFrame(resultados)

display(
    resultados_df.sort_values(
        by="Acurácia média",
        ascending=False
    ).reset_index(drop=True)
)

,Vizinhos (k),Acurácia média,Acurácia (%)
0,20,0.834908,83.490792
1,19,0.834285,83.428487
2,12,0.833662,83.366183
3,18,0.833033,83.303292
4,11,0.832408,83.240792
5,17,0.832406,83.240596
6,13,0.831783,83.178292
7,1,0.831156,83.115596
8,14,0.830533,83.053292
9,15,0.830531,83.053096
